In [1]:
from fontTools.misc.cython import returns
%load_ext autoreload
%autoreload 2

In [2]:

import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion

import json

api_key = os.getenv('EMBEDDING_API_KEY')
collection_name = "vector_db_pro_max"
db_name = "default"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")

SELF_MODEL = "qwen3.6-plus-2026-04-02"
openai = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=api_key,
)


In [18]:
class DocumentChunk(BaseModel):
    source: str = Field(description="文件名")
    document_type: str = Field(description="文件类型")
    original_text: str = Field(description="文档")
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")


In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


In [6]:
split_system_prompt = """
You are a document chunking assistant for a Knowledge Base.
The document is from the shared drive of a company called Insurellm.
The user will provide the document type and its source.

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, ensuring that the entire document is returned in the chunks – do not leave anything out.
There should be overlap between the chunks as appropriate. Typically use about 25% overlap or about 50 words, so that the same text appears in multiple chunks for best retrieval results.

Each chunk should be approximately 200–500 words (or 1–3 natural paragraphs).
Each new chunk should start roughly 75% into the previous chunk (i.e., keep ~25% overlap, or about 50 words).

For each chunk, you must provide:

headline: a brief heading (3–8 words) summarizing the core topic of this chunk – used for fast retrieval.

summary: 1–2 sentences summarizing the key facts, conclusions, or steps in this chunk – helps answer common questions.

original_text: the original text of this chunk from the provided document, exactly as is, not changed in any way.

The user-provided document type and source are only for context to help you chunk appropriately; do not include them in original_text.

Together, your chunks must represent the entire document with overlap.

Output format: JSON.
Example:
{
  "chunks": [
    {
      "headline": "Policy activation conditions",
      "summary": "This chunk explains Insurellm auto insurance policy activation: effective at midnight after full payment, with a 30-day waiting period.",
      "original_text": "The policy becomes effective from 00:00 on the day following receipt of the full premium by the Company... (full paragraph as in original)"
    }
  ]
}
"""

In [7]:
def make_user_prompt(document):
    return f"""
doc_type:{document["type"]}
source:{document["source"]}
text:
{document["text"]}

"""

In [8]:
def make_messages(document):
    return [
        {"role": "system", "content": split_system_prompt},
        {"role": "user", "content": make_user_prompt(document)},
    ]

In [9]:
document_one = documents[2]
messages = make_messages(document_one)


In [10]:
def process_document(document):
    messages = make_messages(document)
    completion = openai.chat.completions.create(
        model=SELF_MODEL,  # 您可以按需更换为其它深度思考模型
        messages=messages,
        response_format={"type": "json_object"},
        extra_body={"enable_thinking": False},
    )
    reply = completion.choices[0].message.content
    result = json.loads(reply)
    chunks = result["chunks"]  # 获取chunks数组
    chunk_objects = []
    for chunk in chunks:
        doc_chunk = DocumentChunk(
            source=document["source"],
            document_type=document["type"],
            original_text=chunk["original_text"],
            headline=chunk["headline"],
            summary=chunk["summary"],
        )
        chunk_objects.append(doc_chunk)

    return chunk_objects

In [13]:
one_doc_chunks = process_document(document_one)
print(document_one["text"])
print("===================================")
print(one_doc_chunks)

# Insurellm Culture

## Vision Statement
To revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.

## Mission Statement
We empower insurance providers and consumers with cutting-edge software solutions that streamline processes, enhance customer experiences, and drive meaningful connections in the insurance marketplace. By combining deep industry expertise with technological innovation, we're building the future of insurance.

## Core Values

### Innovation First
We challenge the status quo and embrace creative problem-solving. Our team is encouraged to experiment, take calculated risks, and push the boundaries of what's possible in insurance technology. We believe that breakthrough solutions come from curiosity, collaboration, and a willingness to learn from both successes and failures.

### Customer Obsession
Our clients' success is our success. We deeply understand our customers' needs and work t

In [11]:
chunks = []  # 所有切分后的结果（持久化）

In [12]:
start_idx = 0  # 起始索引：首次=0，报错后修改为报错索引继续


In [16]:
# ===================== 批量切分主程序（失败立即停止） =====================
# documents = 你的全部文档列表（自行定义）

for i, doc in enumerate(documents):
    # 跳过已处理完成的文档（断点续跑）
    if i < start_idx:
        continue

    print(f"🔄 处理文档：索引 {i}")

    try:
        # 调用你的切分函数
        one_document_chunks = process_document(doc)

        # 追加到总结果（必须用 extend）
        chunks.extend(one_document_chunks)
        print(f"✅ 文档索引 {i} 处理成功\n")

    except Exception as e:
        # 失败：打印索引 + 错误，直接停止程序
        print(f"\n❌ 【致命错误】文档索引 {i} 处理失败！")
        print(f"错误信息：{str(e)}")
        print(f"👉 请修复文档后，将 Cell1 的 start_idx = {i} 重新运行！")

        # 立即停止，不处理下一个文档
        raise

🔄 处理文档：索引 0
✅ 文档索引 0 处理成功

🔄 处理文档：索引 1
✅ 文档索引 1 处理成功

🔄 处理文档：索引 2
✅ 文档索引 2 处理成功

🔄 处理文档：索引 3
✅ 文档索引 3 处理成功

🔄 处理文档：索引 4
✅ 文档索引 4 处理成功

🔄 处理文档：索引 5
✅ 文档索引 5 处理成功

🔄 处理文档：索引 6
✅ 文档索引 6 处理成功

🔄 处理文档：索引 7
✅ 文档索引 7 处理成功

🔄 处理文档：索引 8
✅ 文档索引 8 处理成功

🔄 处理文档：索引 9
✅ 文档索引 9 处理成功

🔄 处理文档：索引 10
✅ 文档索引 10 处理成功

🔄 处理文档：索引 11
✅ 文档索引 11 处理成功

🔄 处理文档：索引 12
✅ 文档索引 12 处理成功

🔄 处理文档：索引 13
✅ 文档索引 13 处理成功

🔄 处理文档：索引 14
✅ 文档索引 14 处理成功

🔄 处理文档：索引 15
✅ 文档索引 15 处理成功

🔄 处理文档：索引 16
✅ 文档索引 16 处理成功

🔄 处理文档：索引 17
✅ 文档索引 17 处理成功

🔄 处理文档：索引 18
✅ 文档索引 18 处理成功

🔄 处理文档：索引 19
✅ 文档索引 19 处理成功

🔄 处理文档：索引 20
✅ 文档索引 20 处理成功

🔄 处理文档：索引 21
✅ 文档索引 21 处理成功

🔄 处理文档：索引 22
✅ 文档索引 22 处理成功

🔄 处理文档：索引 23
✅ 文档索引 23 处理成功

🔄 处理文档：索引 24
✅ 文档索引 24 处理成功

🔄 处理文档：索引 25
✅ 文档索引 25 处理成功

🔄 处理文档：索引 26
✅ 文档索引 26 处理成功

🔄 处理文档：索引 27
✅ 文档索引 27 处理成功

🔄 处理文档：索引 28
✅ 文档索引 28 处理成功

🔄 处理文档：索引 29
✅ 文档索引 29 处理成功

🔄 处理文档：索引 30
✅ 文档索引 30 处理成功

🔄 处理文档：索引 31
✅ 文档索引 31 处理成功

🔄 处理文档：索引 32
✅ 文档索引 32 处理成功

🔄 处理文档：索引 33
✅ 文档索引 33 处理成功

🔄 处理文档：索引 34
✅ 文档索引 34 处理成功

🔄 处理文

In [17]:
#将切分好的数据chunks写入到文件中防止数据丢失
import json
from pathlib import Path

# 保存文件路径（自动存在当前目录，名字可自定义）
SAVE_PATH = "processed_chunks_backup_n1.json"

# 把 DocumentChunk 对象转换成可保存的字典列表
chunks_data = [chunk.model_dump() for chunk in chunks]

# 写入本地文件
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f"✅ 所有chunks已成功保存到本地文件：{SAVE_PATH}")
print(f"📊 总保存数量：{len(chunks_data)} 条")

✅ 所有chunks已成功保存到本地文件：processed_chunks_backup_n.json
📊 总保存数量：380 条


In [19]:
#读取数据

import json

# 读取文件并恢复成对象列表
SAVE_PATH = "processed_chunks_backup_n.json"
with open(SAVE_PATH, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

# 恢复到全局变量 chunks
chunks = [DocumentChunk(**item) for item in chunks_data]

print(f"✅ 从本地文件恢复成功！")
print(f"📊 恢复chunk总数：{len(chunks)} 条")

✅ 从本地文件恢复成功！
📊 恢复chunk总数：380 条


In [20]:
one_chunk = chunks[0]
print(one_chunk)

source='knowledge-base/company/about.md' document_type='company' original_text='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' headline='Insurellm founding and early growth' summary="This chunk details Insurellm's 2015 founding by Avery Lancaster, its initial Markellm product, and rapid expansion to 200 employees and 12 offices by 2020."


In [25]:
import dashscope


def get_embedding(text):
    resp = dashscope.TextEmbedding.call(
        model="text-embedding-v4",
        input=[text],
        api_key=api_key,
        dimension=2048,
    )

    if resp.status_code != 200:
        raise Exception(f"向量生成失败：{resp.message}")

    return resp.output["embeddings"][0]["embedding"]

In [21]:
class EmbeddingDTO:
    def __init__(self, original_text, summary, headline, source, doc_type):
        self.page_content = headline + "\n\n" + summary + "\n\n" + original_text
        self.source = source
        self.doc_type = doc_type

In [22]:
embedding_dto_chunks = [EmbeddingDTO(c.original_text, c.summary, c.headline, c.source, c.document_type) for c in chunks]


In [23]:
embedding_dto_one_chunk = embedding_dto_chunks[0]

In [26]:
one_chunk_embedding = get_embedding(one_chunk.original_text)

In [29]:
from pymilvus import MilvusClient

datas = []
for c in embedding_dto_chunks:
    data = {
        "text": c.page_content,
        "doc_type": c.doc_type,
        "source": c.source,
        "vector": get_embedding(c.page_content)
    }
    datas.append(data)



In [17]:
db_name = "default"
collection_name = "vector_db_pro_max_new_2048_cos_mix"

client = MilvusClient(
    uri="http://192.168.175.142:19530",
    db_name=db_name,
)

client.insert(
    collection_name=collection_name,
    data=datas
)



In [38]:
from week5.milvus_op import get_all
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

all_data = get_all()
milvus_vectors = [v["vector"] for v in all_data]
np_vectors = np.array(milvus_vectors)

milvus_documents = [d["text"] for d in all_data]
milvus_doc_type = [dt["doc_type"] for dt in all_data]
milvus_colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in
                 milvus_doc_type]

# np_vectors = np.array(milvus_vectors, dtype=np.float32)

# 🔍 必须做数据校验！避免数据错误导致卡死
print(f"✅ 原始向量形状: {np_vectors.shape}")  # 必须是 (557, 2048)
print(f"✅ 向量数据类型: {np_vectors.dtype}")  # 应为 float32
print(f"✅ 是否包含NaN: {np.isnan(np_vectors).any()}")  # 必须为 False


✅ 原始向量形状: (380, 2048)
✅ 向量数据类型: float64
✅ 是否包含NaN: False


In [39]:
# ===================== 直接接在你现有代码后运行 =====================
# 1. 优化版 t-SNE（直接用2048维float32向量，无PCA，速度快）
tsne = TSNE(
    n_components=2,
    perplexity=40,  # 适配557条数据，让点更分散
    max_iter=800,  # 充分迭代，优化分布
    init="random",  # 随机初始化，避免点过度集中
    method="barnes_hut",
    angle=0.5,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

# 计算降维结果
reduced_vectors = tsne.fit_transform(np_vectors)

# 🔥 核心修复：添加微小抖动，强行分离重叠的点（解决只显示几个点的问题）
np.random.seed(42)
reduced_vectors += np.random.normal(0, 0.8, reduced_vectors.shape)

# 2. 绘图（完全复用你定义的 milvus_colors / documents / doc_type）
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    # 优化点样式：大尺寸+白色描边，清晰区分重叠点
    marker=dict(size=7, color=milvus_colors, opacity=0.8, line=dict(width=1, color='white')),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

# 修复布局：删除3D专属的scene，2D图正确配置
fig.update_layout(
    title='2D Milvus Vector Visualization',
    title_x=0.5,
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    width=1000,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40),
    showlegend=False  # 你已用颜色区分类型，关闭默认图例更简洁
)

# 显示图像
fig.show()

[t-SNE] Computing 121 nearest neighbors...
[t-SNE] Indexed 380 samples in 0.005s...
[t-SNE] Computed neighbors for 380 samples in 2.598s...
[t-SNE] Computed conditional probabilities for sample 380 / 380
[t-SNE] Mean sigma: 0.337772
[t-SNE] KL divergence after 250 iterations with early exaggeration: 51.344330
[t-SNE] KL divergence after 800 iterations: 0.620104


In [41]:
# 3D t-SNE 降维（优化参数，速度快 + 点分散）
tsne = TSNE(
    n_components=3,
    perplexity=40,  # 适配557条数据，分散点
    max_iter=800,  # 充分迭代
    init="random",
    method="barnes_hut",
    random_state=42,
    verbose=1,
    n_jobs=-1
)
reduced_vectors = tsne.fit_transform(np_vectors)

# 🔥 核心：3D坐标抖动，分离重叠点（解决只显示几个点的问题）
np.random.seed(42)
reduced_vectors += np.random.normal(0, 0.6, reduced_vectors.shape)

# 3D散点图（复用你所有变量）
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    # 优化样式：白色描边，区分重叠点
    marker=dict(size=6, color=milvus_colors, opacity=0.8, line=dict(width=1, color='white')),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

# 3D布局（保留你的原版配置，无修改）
fig.update_layout(
    title='3D Milvus Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

[t-SNE] Computing 121 nearest neighbors...
[t-SNE] Indexed 380 samples in 0.002s...
[t-SNE] Computed neighbors for 380 samples in 0.028s...
[t-SNE] Computed conditional probabilities for sample 380 / 380
[t-SNE] Mean sigma: 0.337772
[t-SNE] KL divergence after 250 iterations with early exaggeration: 51.388447
[t-SNE] KL divergence after 750 iterations: 0.642425


### 进一步提升rag检索指标：创建混合索引

In [36]:
import jieba

db_name = "default"
old_collection_name = "vector_db_pro_max_old_2048_cos"
new_collection_name = "vector_db_pro_max_old_2048_cos_mix"
DIMENSION = 2048

client = MilvusClient(
    uri="http://192.168.175.142:19530",
    db_name=db_name,
)


def text_to_sparse_vector(text: str):
    """
    中文文本 → Milvus 稀疏向量
    作用：做关键词检索（替代BM25，数据库原生支持）
    """
    # 中文分词
    words = list(jieba.cut(text))
    # 词频统计
    word_count = {}
    for w in words:
        if len(w.strip()) > 0:
            word_count[w] = word_count.get(w, 0) + 1

    # 转成稀疏向量格式（Milvus要求：{索引: 权重}）
    sparse_dict = {}
    for idx, (word, cnt) in enumerate(word_count.items()):
        sparse_dict[idx] = float(cnt)

    return sparse_dict


def get_all(collection, fields=None):
    if fields is not None and not isinstance(fields, list):
        raise ValueError("参数错误！请传入字段列表，例如：['a']、['a', 'b']")

    all_results = []

    batch_size = 16384
    query_param = {
        "collection_name": collection,
        "filter": "",
        "batch_size": batch_size,
        "output_fields": fields if fields else ["*"]
    }

    iterator = client.query_iterator(**query_param)
    while True:
        res = iterator.next()

        if not res:
            iterator.close()
            break

        all_results.extend(res)

    return all_results


def insert(collection, data: list[dict]):
    if not isinstance(data, list):
        raise TypeError("插入失败！data 必须是列表，格式：[{}, {}]")

    # 强制校验：列表里的每个元素必须是字典
    for item in data:
        if not isinstance(item, dict):
            raise TypeError("插入失败！列表内必须是字典，格式：[{...}, {...}]")
    client.insert(
        collection_name=collection,
        data=data
    )

In [30]:

learn_text_list = [item.page_content for item in embedding_dto_chunks]



In [31]:
from pymilvus.model.sparse import BM25EmbeddingFunction
from pymilvus.model.sparse.bm25 import build_default_analyzer

analyzer = build_default_analyzer(language="en")  # 如果是中文，可能需要换成jieba或其他分词器
bm25_ef = BM25EmbeddingFunction(analyzer)

# 3. 拟合语料
bm25_ef.fit(learn_text_list)

In [32]:
import pickle

with open("bm25_encoder_new.pkl", "wb") as f:
    pickle.dump(bm25_ef, f)

In [33]:
one_sp_v = bm25_ef.encode_documents(learn_text_list[0])

In [32]:
one_sp_v_data = {i: v for i, v in zip(one_sp_v.indices, one_sp_v.data)}


In [34]:
insert_datas = []

for item in datas:
    # 输入必须是列表形式，即使只有1个文本
    one_encode_data_list = bm25_ef.encode_documents([item["text"]])
    one_sp_v = one_encode_data_list[0]  # 这是一个 coo_array

    # 转换为字典（coo_array 的属性是 .col 和 .data）
    one_sp_v_data = {int(i): float(v) for i, v in zip(one_sp_v.col, one_sp_v.data)}

    insert_data = {
        "text": item["text"],
        "doc_type": item["doc_type"],
        "source": item["source"],
        "vector": item["vector"],
        "sparse_vector": one_sp_v_data

    }
    insert_datas.append(insert_data)

In [37]:
insert("vector_db_pro_max_new_2048_cos_mix", insert_datas)
